In [1]:
%pip install beautifulsoup4


   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
from bs4 import BeautifulSoup

URL_LISTADO = "https://repositorioelectoral-uba.ar/afiches/"
pagina = requests.get(URL_LISTADO, timeout=30)
print(pagina.status_code)

200


In [3]:
soup = BeautifulSoup(pagina.text, "html.parser")
titulos = soup.find_all("h2", class_="entry-title")
print(len(titulos))

for h2 in titulos:
    enlace = h2.find("a")
    print(enlace.text, "->", enlace["href"])

12
PROGRESISTAS | Margarita Stolbizer – Miguel Angel Olaviaga -> https://repositorioelectoral-uba.ar/afiches/progresistas-margarita-stolbizer-miguel-angel-olaviaga/
UNA NUEVA ALTERNATIVA | Julio Zamora -> https://repositorioelectoral-uba.ar/afiches/una-nueva-alternativa-julio-zamora/
UNA NUEVA ALTERNATIVA | Segio Massa – Gustavo Saenz -> https://repositorioelectoral-uba.ar/afiches/una-nueva-alternativa-segio-massa-gustavo-saenz/
COMPROMISO FEDERAL | Adolfo Rodriguez Saa – Liliana Negre de Alonso -> https://repositorioelectoral-uba.ar/afiches/compromiso-federal-adolfo-rodriguez-saa-liliana-negre-de-alonso/
FRENTE DE IZQUIERDA Y DE LOS TRABAJADORES | Christian Castillo -> https://repositorioelectoral-uba.ar/afiches/frente-de-izquierda-y-de-los-trabajadores-christian-castillo/
FRENTE DE IZQUIERDA Y DE LOS TRABAJADORES | Nicolas del Caño – Myriam Bregman -> https://repositorioelectoral-uba.ar/afiches/frente-de-izquierda-y-de-los-trabajadores-nicolas-del-cano-myriam-bregman/
FPV | Horacio G

In [6]:
import time

fichas = []
for numero in range(1, 5):
    if numero == 1:
        url = URL_LISTADO
    else:
        url = URL_LISTADO + f"page/{numero}/"
    pagina = requests.get(url, timeout=30)
    soup = BeautifulSoup(pagina.text, "html.parser")

    for articulo in soup.find_all("article", class_="entry-card"):
        enlace = articulo.find("h2").find("a")
        partido, formula = enlace.text.split("|")
        fichas.append({
            "partido": partido.strip(),
            "formula": formula.strip(),
            "anio": articulo.find("time")["datetime"][:4],
            "url_ficha": enlace["href"],
        })
    time.sleep(1)

print(len(fichas))
print(fichas[0])

38
{'partido': 'PROGRESISTAS', 'formula': 'Margarita Stolbizer – Miguel Angel Olaviaga', 'anio': '2015', 'url_ficha': 'https://repositorioelectoral-uba.ar/afiches/progresistas-margarita-stolbizer-miguel-angel-olaviaga/'}


In [7]:
ficha = fichas[0]
pagina = requests.get(ficha["url_ficha"], timeout=30)
soup = BeautifulSoup(pagina.text, "html.parser")

contenido = soup.find(class_="entry-content")
for img in contenido.find_all("img"):
    url_imagen = img.find_parent("a")["href"]
    print(url_imagen)

https://repositorioelectoral-uba.ar/wp-content/uploads/2023/09/IMG_0030.jpg
https://repositorioelectoral-uba.ar/wp-content/uploads/2023/09/IMG_0031.jpg
https://repositorioelectoral-uba.ar/wp-content/uploads/2023/09/IMG_0033.jpg
https://repositorioelectoral-uba.ar/wp-content/uploads/2023/09/IMG_0034.jpg
https://repositorioelectoral-uba.ar/wp-content/uploads/2023/09/IMG_0035.jpg


In [8]:
imagenes = []
for i, ficha in enumerate(fichas, start=1):
    id_campana = f"f{i:02d}"
    pagina = requests.get(ficha["url_ficha"], timeout=30)
    soup = BeautifulSoup(pagina.text, "html.parser")
    contenido = soup.find(class_="entry-content")

    for j, img in enumerate(contenido.find_all("img"), start=1):
        imagenes.append({
            "id_afiche": f"{id_campana}_{j}",
            "id_campana": id_campana,
            "partido": ficha["partido"],
            "formula": ficha["formula"],
            "anio": ficha["anio"],
            "url": img.find_parent("a")["href"],
        })
    time.sleep(1)

print(len(imagenes))
print(imagenes[0])

141
{'id_afiche': 'f01_1', 'id_campana': 'f01', 'partido': 'PROGRESISTAS', 'formula': 'Margarita Stolbizer – Miguel Angel Olaviaga', 'anio': '2015', 'url': 'https://repositorioelectoral-uba.ar/wp-content/uploads/2023/09/IMG_0030.jpg'}


In [9]:
import pandas as pd

tabla = pd.DataFrame(imagenes)
print(tabla.shape)
tabla.head()

(141, 6)


,id_afiche,id_campana,partido,formula,anio,url
0,f01_1,f01,PROGRESISTAS,Margarita Stolbizer – Miguel Angel Olaviaga,2015,https://repositorioelectoral-uba.ar/wp-content...
1,f01_2,f01,PROGRESISTAS,Margarita Stolbizer – Miguel Angel Olaviaga,2015,https://repositorioelectoral-uba.ar/wp-content...
2,f01_3,f01,PROGRESISTAS,Margarita Stolbizer – Miguel Angel Olaviaga,2015,https://repositorioelectoral-uba.ar/wp-content...
3,f01_4,f01,PROGRESISTAS,Margarita Stolbizer – Miguel Angel Olaviaga,2015,https://repositorioelectoral-uba.ar/wp-content...
4,f01_5,f01,PROGRESISTAS,Margarita Stolbizer – Miguel Angel Olaviaga,2015,https://repositorioelectoral-uba.ar/wp-content...


In [10]:
import os

os.makedirs("imagenes", exist_ok=True)

archivos = []
for fila in tabla.itertuples():
    extension = os.path.splitext(fila.url)[1]
    archivo = f"imagenes/{fila.id_afiche}{extension}"
    if not os.path.exists(archivo):
        respuesta = requests.get(fila.url, timeout=60)
        respuesta.raise_for_status()
        with open(archivo, "wb") as f:
            f.write(respuesta.content)
        time.sleep(1)
    archivos.append(archivo)

tabla["archivo"] = archivos
print(len(os.listdir("imagenes")))

141


In [11]:
tabla.to_csv("afiches.csv", index=False)
print(tabla.shape)

(141, 7)
